<a href="https://colab.research.google.com/github/SantiagoAmaya99/Crack-Detector-Using-Deep-Learning-Models/blob/main/Prueba_de_modelos_Detecci%C3%B3n_de_grietas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd '/content/drive/MyDrive/Dataset_IA2'

Mounted at /content/drive
/content/drive/MyDrive/Dataset_IA2


In [ ]:
#@title Librerias
import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models
from sklearn.model_selection import StratifiedGroupKFold
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications import ConvNeXtTiny
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import os, re, shutil
from pathlib import Path
from tqdm.auto import tqdm

In [ ]:
!cp "/content/drive/MyDrive/Dataset_IA2_C.zip" "/content/"

In [ ]:
!unzip -q "/content/Dataset_IA2_C.zip" -d "/content/dataset_crudo"

In [ ]:
import pandas as pd
import re
from pathlib import Path
from tqdm import tqdm

# Apuntamos a la carpeta que ya está en el SSD de Colab
RUTA_CRUDO = Path('/content/dataset_crudo')

def id_foto(nombre_archivo):
    m = re.search(r'(\d+)[-_]\d+', Path(nombre_archivo).stem)
    return m.group(1) if m else Path(nombre_archivo).stem

registros = []

for material in ['Decks', 'Pavements', 'Walls']:
    for clase, etiqueta in [('Cracked', 1), ('Non-cracked', 0)]:
        carpeta_origen = RUTA_CRUDO / material / clase

        if carpeta_origen.exists():
            # Atrapa todo sin importar minúsculas/mayúsculas en la extensión
            archivos = [
            f for f in carpeta_origen.iterdir()
            if f.is_file()
            and f.suffix.lower() in ['.jpg', '.jpeg', '.png']
            and f.stat().st_size > 0
             ]

            for f in tqdm(archivos, desc=f'{material}/{clase}'):
                # Guardamos directamente la ruta original, cero copias
                registros.append({
                    'ruta': str(f),
                    'etiqueta': etiqueta,
                    'grupo': f'{material}_{id_foto(f.name)}'
                })

df = pd.DataFrame(registros)
print(f"Total de imágenes mapeadas correctamente: {len(df)}")

Walls/Non-cracked: 100%|██████████| 1475/1475 [00:00<00:00, 154117.84it/s]

Total de imágenes mapeadas correctamente: 43280


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

seed = 42

def dividir_por_grupo(datos, fraccion, seed):
    gss = GroupShuffleSplit(n_splits=1, test_size=fraccion, random_state=seed)
    idx_a, idx_b = next(gss.split(datos, groups=datos['grupo']))
    return datos.iloc[idx_a].reset_index(drop=True), datos.iloc[idx_b].reset_index(drop=True)

df_trainval, df_test = dividir_por_grupo(df, 0.15, seed)
df_train, df_val = dividir_por_grupo(df_trainval, 0.15 / 0.85, seed)

print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

Train: 30128 | Val: 7016 | Test: 6136


In [ ]:
import tensorflow as tf
from tensorflow import keras

seed = 42


TAM_IMG = (128, 128)
TAM_LOTE = 64
AUTOTUNE = tf.data.AUTOTUNE

n_total = len(df_train)
n1 = int(df_train.etiqueta.sum())
n0 = n_total - n1
PESO_CLASE = {0: n_total / (2 * n0), 1: n_total / (2 * n1)}

augment = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.1, fill_mode='reflect'),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2, value_range=(0, 255)),
])

def cargar_imagen(ruta, etiqueta):
    img = tf.io.read_file(ruta)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, TAM_IMG)
    return img, tf.cast(etiqueta, tf.float32)

def crear_dataset(datos, entrenamiento=False):
    rutas = datos['ruta'].astype(str).values
    etiquetas = datos['etiqueta'].values.astype('int32')
    ds = tf.data.Dataset.from_tensor_slices((rutas, etiquetas))

    if entrenamiento:
        ds = ds.shuffle(len(datos), seed, reshuffle_each_iteration=True)

    ds = ds.map(cargar_imagen, num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (x / 255.0, y), num_parallel_calls=AUTOTUNE)

    if entrenamiento:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
        w0, w1 = PESO_CLASE[0], PESO_CLASE[1]
        ds = ds.map(lambda x, y: (x, y, tf.where(y > 0.5, w1, w0)), num_parallel_calls=AUTOTUNE)

    return ds.batch(TAM_LOTE).prefetch(AUTOTUNE)

ds_train = crear_dataset(df_train, entrenamiento=True)
ds_val = crear_dataset(df_val)
ds_test = crear_dataset(df_test)

In [ ]:
import os

# Filtrar imágenes que tengan tamaño 0 o que no existan
def es_valida(ruta):
    p = Path(ruta)
    return p.exists() and p.stat().st_size > 0

# Limpiar el DataFrame original
df_limpio = df[df['ruta'].apply(es_valida)].reset_index(drop=True)
print(f"Imágenes válidas: {len(df_limpio)} de {len(df)}")

Imágenes válidas: 43280 de 43281


In [ ]:
import tensorflow as tf

def obtener_callbacks(nombre_modelo):

    return [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_auc',
            mode='max',
            patience=8,
            restore_best_weights=True,
            verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=f'{nombre_modelo}_feature_extraction.keras',
            monitor='val_auc',
            mode='max',
            save_best_only=True,
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1
        )
    ]

#ResNET 50

In [ ]:
#@title Modelo base
def build_resnet50_base(input_shape=(128, 128, 3)):
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )

    # 2. Construir el nuevo modelo agregando la cabeza personalizada
    modelResNET = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4), # Requisito de regularización base para los experimentos
        layers.Dense(1, activation='sigmoid') # Salida binaria (grieta / no grieta)
    ])

    return modelResNET, base_model

# Crear la arquitectura inicial
modelResNET, base_model = build_resnet50_base()
modelResNET.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 4, 4, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         2,049 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,589,761 (89.99 MB)

 Trainable params: 23,536,641 (89.79 MB)

 Non-trainable params: 53,120 (207.50 KB)

In [ ]:
history_resnet = modelResNET.fit(
    ds_train,
    validation_data=ds_val,
    epochs=20,
    callbacks=obtener_callbacks('resnet50')
)

Epoch 1/20
471/471 ━━━━━━━━━━━━━━━━━━━━ 0s 324ms/step - accuracy: 0.5135 - auc: 0.5005 - loss: 0.6941
Epoch 1: val_auc improved from None to 0.67540, saving model to resnet50_feature_extraction.h5



Epoch 1: finished saving model to resnet50_feature_extraction.h5
471/471 ━━━━━━━━━━━━━━━━━━━━ 187s 371ms/step - accuracy: 0.5165 - auc: 0.5060 - loss: 0.6944 - val_accuracy: 0.1943 - val_auc: 0.6754 - val_loss: 0.7318 - learning_rate: 0.0010
Epoch 2/20
471/471 ━━━━━━━━━━━━━━━━━━━━ 0s 319ms/step - accuracy: 0.4296 - auc: 0.5215 - loss: 0.6982
Epoch 2: val_auc improved from 0.67540 to 0.68825, saving model to resnet50_feature_extraction.h5



Epoch 2: finished saving model to resnet50_feature_extraction.h5
471/471 ━━━━━━━━━━━━━━━━━━━━ 163s 346ms/step - accuracy: 0.5069 - auc: 0.5270 - loss: 0.6921 - val_accuracy: 0.2141 - val_auc: 0.6882 - val_loss: 0.7001 - learning_rate: 0.0010
Epoch 3/20
471/471 ━━━━━━━━━━━━━━━━━━━━ 0s 329ms/step - accuracy: 0.5067 - auc: 0.5176 - loss: 0.6953
Epoch 3: val_auc improved from 0.68825 to 0.69778, saving model to resnet50_feature_extraction.h5



Epoch 3: finished saving model to resnet50_feature_extraction.h5
471/471 ━━━━━━━━━━━━━━━━━━━━ 165s 349ms/step - accuracy: 0.5335 - auc: 0.5191 - loss: 0.6930 - val_accuracy: 0.8066 - val_auc: 0.6978 - val_loss: 0.6837 - learning_rate: 0.0010
Epoch 4/20
471/471 ━━━━━━━━━━━━━━━━━━━━ 0s 325ms/step - accuracy: 0.4802 - auc: 0.5185 - loss: 0.6920
Epoch 4: val_auc improved from 0.69778 to 0.70533, saving model to resnet50_feature_extraction.h5



Epoch 4: finished saving model to resnet50_feature_extraction.h5
471/471 ━━━━━━━━━━━━━━━━━━━━ 163s 345ms/step - accuracy: 0.5092 - auc: 0.5193 - loss: 0.6928 - val_accuracy: 0.5309 - val_auc: 0.7053 - val_loss: 0.6918 - learning_rate: 0.0010
Epoch 5/20
471/471 ━━━━━━━━━━━━━━━━━━━━ 0s 321ms/step - accuracy: 0.5088 - auc: 0.5336 - loss: 0.6946
Epoch 5: val_auc improved from 0.70533 to 0.70637, saving model to resnet50_feature_extraction.h5



Epoch 5: finished saving model to resnet50_feature_extraction.h5
471/471 ━━━━━━━━━━━━━━━━━━━━ 161s 342ms/step - accuracy: 0.5364 - auc: 0.5303 - loss: 0.6918 - val_accuracy: 0.8072 - val_auc: 0.7064 - val_loss: 0.6500 - learning_rate: 0.0010
Epoch 6/20
471/471 ━━━━━━━━━━━━━━━━━━━━ 0s 319ms/step - accuracy: 0.5790 - auc: 0.5344 - loss: 0.6840
Epoch 6: val_auc improved from 0.70637 to 0.70814, saving model to resnet50_feature_extraction.h5


... (2 outputs left out)


Epoch 9: finished saving model to resnet50_feature_extraction.h5
471/471 ━━━━━━━━━━━━━━━━━━━━ 160s 340ms/step - accuracy: 0.5467 - auc: 0.5462 - loss: 0.6888 - val_accuracy: 0.7996 - val_auc: 0.7174 - val_loss: 0.6779 - learning_rate: 5.0000e-04
Epoch 10/20
471/471 ━━━━━━━━━━━━━━━━━━━━ 0s 317ms/step - accuracy: 0.4892 - auc: 0.5449 - loss: 0.6915
Epoch 10: val_auc improved from 0.71738 to 0.71741, saving model to resnet50_feature_extraction.h5



Epoch 10: finished saving model to resnet50_feature_extraction.h5
471/471 ━━━━━━━━━━━━━━━━━━━━ 159s 338ms/step - accuracy: 0.5083 - auc: 0.5447 - loss: 0.6895 - val_accuracy: 0.8093 - val_auc: 0.7174 - val_loss: 0.6538 - learning_rate: 5.0000e-04
Epoch 11/20
471/471 ━━━━━━━━━━━━━━━━━━━━ 0s 317ms/step - accuracy: 0.5174 - auc: 0.5416 - loss: 0.6945
Epoch 11: val_auc did not improve from 0.71741

Epoch 11: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
471/471 ━━━━━━━━━━━━━━━━━━━━ 157s 332ms/step - accuracy: 0.5414 - auc: 0.5421 - loss: 0.6894 - val_accuracy: 0.2996 - val_auc: 0.7172 - val_loss: 0.7029 - learning_rate: 5.0000e-04
Epoch 12/20
471/471 ━━━━━━━━━━━━━━━━━━━━ 0s 318ms/step - accuracy: 0.4975 - auc: 0.5514 - loss: 0.6883
Epoch 12: val_auc did not improve from 0.71741
471/471 ━━━━━━━━━━━━━━━━━━━━ 157s 333ms/step - accuracy: 0.5199 - auc: 0.5534 - loss: 0.6881 - val_accuracy: 0.6005 - val_auc: 0.7143 - val_loss: 0.6877 - learning_rate: 2.5000e-04
Epoch 13/20



Epoch 13: finished saving model to resnet50_feature_extraction.h5
471/471 ━━━━━━━━━━━━━━━━━━━━ 162s 343ms/step - accuracy: 0.5265 - auc: 0.5566 - loss: 0.6877 - val_accuracy: 0.8030 - val_auc: 0.7185 - val_loss: 0.6681 - learning_rate: 2.5000e-04
Epoch 14/20
471/471 ━━━━━━━━━━━━━━━━━━━━ 0s 317ms/step - accuracy: 0.5484 - auc: 0.5420 - loss: 0.6852
Epoch 14: val_auc did not improve from 0.71845

Epoch 14: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
471/471 ━━━━━━━━━━━━━━━━━━━━ 157s 332ms/step - accuracy: 0.5282 - auc: 0.5503 - loss: 0.6882 - val_accuracy: 0.7712 - val_auc: 0.7181 - val_loss: 0.6800 - learning_rate: 2.5000e-04
Epoch 15/20
471/471 ━━━━━━━━━━━━━━━━━━━━ 0s 315ms/step - accuracy: 0.4610 - auc: 0.5556 - loss: 0.6939
Epoch 15: val_auc improved from 0.71845 to 0.72027, saving model to resnet50_feature_extraction.h5



Epoch 15: finished saving model to resnet50_feature_extraction.h5
471/471 ━━━━━━━━━━━━━━━━━━━━ 160s 340ms/step - accuracy: 0.5090 - auc: 0.5557 - loss: 0.6882 - val_accuracy: 0.8070 - val_auc: 0.7203 - val_loss: 0.6608 - learning_rate: 1.2500e-04
Epoch 16/20
471/471 ━━━━━━━━━━━━━━━━━━━━ 0s 322ms/step - accuracy: 0.5508 - auc: 0.5529 - loss: 0.6902
Epoch 16: val_auc did not improve from 0.72027
471/471 ━━━━━━━━━━━━━━━━━━━━ 160s 340ms/step - accuracy: 0.5443 - auc: 0.5528 - loss: 0.6885 - val_accuracy: 0.8047 - val_auc: 0.7197 - val_loss: 0.6732 - learning_rate: 1.2500e-04
Epoch 17/20
471/471 ━━━━━━━━━━━━━━━━━━━━ 0s 354ms/step - accuracy: 0.5215 - auc: 0.5480 - loss: 0.6893
Epoch 17: val_auc did not improve from 0.72027

Epoch 17: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.
471/471 ━━━━━━━━━━━━━━━━━━━━ 174s 369ms/step - accuracy: 0.5173 - auc: 0.5507 - loss: 0.6886 - val_accuracy: 0.8032 - val_auc: 0.7166 - val_loss: 0.6696 - learning_rate: 1.2500e-04
Epoch 18/20
4

#EfficientNetB0

In [ ]:
def build_efficientnet_base(input_shape=(128, 128, 3)):
    # 1. Instanciar EfficientNetB0 preentrenada en ImageNet
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )

    # 2. Construir la cabeza personalizada
    model_efficient = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4), # Regularización requerida
        layers.Dense(1, activation='sigmoid') # Clasificación binaria
    ])

    return model_efficient, base_model

model, base_model = build_efficientnet_base()

In [ ]:
history_efficientnet = model_efficientnet.fit(
    ds_train,
    validation_data=ds_val,
    epochs=20,
    callbacks=obtener_callbacks('efficientnet_b0')
)

#VGG16

In [ ]:

def build_vgg16_base(input_shape=(128, 128, 3)):
    # 1. Cargar VGG16 sin la cabeza
    base_vgg = VGG16(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    base_vgg.trainable = False

    model_vgg = models.Sequential([
        base_vgg,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid') # Salida binaria
    ])

    return model_vgg, base_vgg

# Crear la arquitectura inicial
model_vgg, base_vgg = build_vgg16_base()
model_vgg.summary()

In [ ]:
model_vgg.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)
history_vgg = model_vgg.fit(
    ds_train,
    validation_data=ds_val,
    epochs=20,
    callbacks=obtener_callbacks('vgg16')
)

#DenseNet121

In [ ]:
# 1. Base y arquitectura
base_dense = DenseNet121(weights='imagenet', include_top=False, input_shape=(128, 128, 3))

base_dense.trainable = False

model_dense = models.Sequential([
    base_dense,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid')
])

# 3. Compilación (LR = 1e-4)
model_dense.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

model_dense.summary()


In [ ]:
history_dense = model_dense.fit(
    ds_train,
    validation_data=ds_val,
    epochs=20,
    callbacks=obtener_callbacks('densenet121')
)

#ConvNeXt

In [ ]:
base_convnext = ConvNeXtTiny(weights='imagenet', include_top=False, input_shape=(128, 128, 3))

base_convnext.trainable = False

model_convnext = models.Sequential([
    base_convnext,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4), # Igualado a 0.4
    layers.Dense(1, activation='sigmoid')
])

model_convnext.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

model_convnext.summary()



In [ ]:
history_dense = model_dense.fit(
    ds_train,
    validation_data=ds_val,
    epochs=20,
    callbacks=obtener_callbacks('densenet121')
)

#MobileNetV3

In [ ]:

def build_mobilenetv2_base(input_shape=(128, 128, 3)):
    base_mobilenet = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )

    base_mobilenet.trainable = False

    model_mobilenet = models.Sequential([
        base_mobilenet,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid')
    ])

    return model_mobilenet, base_mobilenet

# Crear el modelo
model_mobilenet, base_mobilenet = build_mobilenetv2_base()

model_mobilenet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

model_mobilenet.summary()

In [ ]:
history_dense = model_dense.fit(
    ds_train,
    validation_data=ds_val,
    epochs=20,
    callbacks=obtener_callbacks('densenet121')
)